# 05 · Random Forest

Hundreds of decision trees, each grown on a bootstrap sample of women and a random share of the predictors at every split, averaged. It needs no scaling and finds interactions by itself.

**Preparation.** Numbers: median fill. Categories: the most frequent level fills a missing value, then each level becomes an integer code the trees split on.

**Tuning.** The study fixed a small grid of settings in advance (the grid below). Every
setting is fitted on four of the five development folds and scored on the fifth, in turn, and
the settings are ranked by **survey-weighted average precision, out of fold**. Inside each
fold the survey weights are rescaled to average one, so their size does not interact with the
model's regularisation. The best setting is refitted on all 5,518 development women, and the
held-out women are scored **once**. The operating threshold (for sensitivity and specificity)
is chosen on the development predictions only.

The model is tuned separately at each rung: three fits, each saved to
`results/s21/<rung>/Random Forest.npz` with its out-of-fold and held-out predictions and the setting
it chose. A fit already saved is skipped, so the notebook can be stopped and resumed.

In [ ]:
# Settings for this notebook
QUICK = False   # True: fewer resamples for the intervals (a check that everything runs)

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib

MODEL = "Random Forest"
FOLD_JOBS = 5   # folds fitted in parallel; lower it on a machine with few cores

In [ ]:
grid = lib.GRIDS[MODEL]
print(pd.DataFrame([{"setting": k, "values": ", ".join("none" if v is None else str(v) for v in vals)}
                    for k, vals in grid.items()]).to_string(index=False))
print(f"\n{len(lib.grid_settings(MODEL))} settings, each scored at every rung")

In [ ]:
for r in lib.RUNGS:
    job = lib.Job("grid", "s21", r, MODEL, MODEL)
    if lib.is_done(P.results, job):
        print(f"{job.id}: already done")
        continue
    lib.run(job, P.results, fold_jobs=FOLD_JOBS)

## What the grid chose, and the check against the study

In [ ]:
for r in lib.RUNGS:
    meta = lib.load_cell(P.results, "s21", r, MODEL)["meta"]
    print(f"{r}: {meta['params']}  (development AP {meta['dev_metrics']['AP']:.3f}, held-out AP {meta['held_metrics']['AP']:.3f})")
repro.compare_cells(P.results, [MODEL])
repro.report()